# Adaptive Knowledge Boundary Testing with TextGrad

Systematically probe a student LLM's knowledge boundaries by adaptively generating questions until failure.

**Research goal:** Map competence frontiers in a subject area

**Method:** Teacher LLM adaptively generates questions, starting broad and drilling deeper until student fails

In [ ]:
!pip install textgrad matplotlib -q
print('Dependencies installed')

In [ ]:
import textgrad as tg
import os
import re
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple, Optional

In [ ]:
# Configure API key
OPENROUTER_API_KEY = "your-key-here"
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

print('API key configured')

In [ ]:
# Model configuration
TEACHER_MODEL = "experimental:openrouter/anthropic/claude-3.5-sonnet"
STUDENT_MODEL = "experimental:openrouter/meta-llama/llama-3.2-3b-instruct:free"

# Test configuration
SUBJECT_AREA = "Python programming"
MAX_QUESTIONS = 15
FAIL_THRESHOLD = 0.6

tg.set_backward_engine(TEACHER_MODEL, override=True, cache=True)

print(f'Teacher: {TEACHER_MODEL}')
print(f'Student: {STUDENT_MODEL}')
print(f'Subject: {SUBJECT_AREA}')
print(f'Fail threshold: {FAIL_THRESHOLD}')

In [ ]:
@dataclass
class Question:
    text: str
    difficulty: int
    topic: str
    subtopic: str
    expected_answer: str = ""

@dataclass
class Response:
    question_id: int
    answer: str
    score: float
    correct: bool
    
print('Data structures defined')

In [ ]:
class QuestionGenerator:
    def __init__(self, subject: str, teacher_model: str):
        self.subject = subject
        self.teacher_llm = tg.BlackboxLLM(teacher_model)
        self.difficulty_history = []
        self.topic_coverage = {}
        
    def generate(self, history: List[Response]) -> Question:
        strategy = self._get_strategy(history)
        
        prompt = f"""Generate a test question for {self.subject}.\n\n{strategy}\n\nOutput format:\nDIFFICULTY: <1-5>\nTOPIC: <main topic>\nSUBTOPIC: <specific area>\nQUESTION: <question>\nEXPECTED: <brief answer>"""
        
        q_var = tg.Variable(prompt, requires_grad=False)
        response = self.teacher_llm(q_var)
        
        return self._parse(response.value)
    
    def _get_strategy(self, history: List[Response]) -> str:
        if not history:
            return "First question. Start at difficulty 1."
        
        recent = history[-3:]
        avg = np.mean([r.score for r in recent])
        
        if avg >= 0.85:
            target = min(5, self.difficulty_history[-1] + 1 if self.difficulty_history else 2)
            strategy = f"Student performing well. Increase difficulty to {target}."
        elif avg < 0.6:
            target = max(1, self.difficulty_history[-1] - 1 if self.difficulty_history else 1)
            strategy = f"Student struggling. Probe gap at difficulty {target}."
        else:
            target = self.difficulty_history[-1] if self.difficulty_history else 2
            strategy = f"Mixed performance. Maintain difficulty {target}."
        
        self.difficulty_history.append(target)
        return strategy
    
    def _parse(self, text: str) -> Question:
        def extract(field):
            pattern = f"{field}:\\s*(.+?)(?:\\n|$)"
            match = re.search(pattern, text, re.IGNORECASE)
            return match.group(1).strip() if match else ""
        
        try:
            difficulty = int(extract("DIFFICULTY"))
        except:
            difficulty = 2
        
        topic = extract("TOPIC")
        if topic:
            self.topic_coverage[topic] = self.topic_coverage.get(topic, 0) + 1
        
        return Question(
            text=extract("QUESTION"),
            difficulty=difficulty,
            topic=topic,
            subtopic=extract("SUBTOPIC"),
            expected_answer=extract("EXPECTED")
        )

print('QuestionGenerator defined')

In [ ]:
class Grader:
    def __init__(self, teacher_model: str):
        self.teacher_llm = tg.BlackboxLLM(teacher_model)
    
    def grade(self, question: Question, answer: str) -> Tuple[float, str]:
        prompt = f"""Grade this answer.\n\nQuestion: {question.text}\nExpected: {question.expected_answer}\nStudent: {answer}\n\nOutput:\nSCORE: <0.0-1.0>\nREASONING: <brief>"""
        
        g_var = tg.Variable(prompt, requires_grad=False)
        evaluation = self.teacher_llm(g_var)
        
        score_match = re.search(r"SCORE:\\s*([\\d.]+)", evaluation.value)
        score = float(score_match.group(1)) if score_match else 0.5
        
        reasoning_match = re.search(r"REASONING:\\s*(.+)", evaluation.value, re.IGNORECASE | re.DOTALL)
        reasoning = reasoning_match.group(1).strip() if reasoning_match else ""
        
        return score, reasoning

print('Grader defined')

In [ ]:
class TestSession:
    def __init__(self, subject: str, student_model: str, teacher_model: str):
        self.subject = subject
        self.student_llm = tg.BlackboxLLM(student_model)
        self.generator = QuestionGenerator(subject, teacher_model)
        self.grader = Grader(teacher_model)
        self.questions = []
        self.responses = []
        self.failure_point = None
    
    def run(self, max_questions: int, fail_threshold: float):
        print(f'Testing: {self.subject}\\n')
        
        for i in range(max_questions):
            print(f'Question {i+1}/{max_questions}')
            
            question = self.generator.generate(self.responses)
            self.questions.append(question)
            
            print(f'  Difficulty: {question.difficulty}/5')
            print(f'  Topic: {question.topic}')
            print(f'  Q: {question.text[:80]}...')
            
            q_var = tg.Variable(question.text, requires_grad=False)
            answer = self.student_llm(q_var)
            print(f'  A: {answer.value[:80]}...')
            
            score, reasoning = self.grader.grade(question, answer.value)
            correct = score >= fail_threshold
            
            print(f'  Score: {score:.3f} ({"PASS" if correct else "FAIL"})')
            
            response = Response(
                question_id=i,
                answer=answer.value,
                score=score,
                correct=correct
            )
            self.responses.append(response)
            
            if not correct:
                self.failure_point = i
                print(f'\\nKnowledge boundary found at question {i+1}')
                print(f'Topic: {question.topic}, Difficulty: {question.difficulty}/5')
                break
        
        return self.responses

print('TestSession defined')

In [ ]:
def plot_results(session: TestSession):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    questions_range = list(range(1, len(session.responses) + 1))
    scores = [r.score for r in session.responses]
    
    # Score trajectory
    ax = axes[0, 0]
    colors = ['green' if r.correct else 'red' for r in session.responses]
    ax.scatter(questions_range, scores, c=colors, s=100, alpha=0.6, edgecolors='black')
    ax.plot(questions_range, scores, 'k-', alpha=0.3)
    ax.axhline(y=0.6, color='red', linestyle='--', linewidth=2, label='Fail threshold')
    ax.set_xlabel('Question number')
    ax.set_ylabel('Score')
    ax.set_title('Score trajectory')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Difficulty progression
    ax = axes[0, 1]
    difficulties = [q.difficulty for q in session.questions]
    ax.plot(questions_range, difficulties, marker='o', linewidth=2)
    if session.failure_point is not None:
        ax.axvline(x=session.failure_point+1, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel('Question number')
    ax.set_ylabel('Difficulty (1-5)')
    ax.set_title('Difficulty progression')
    ax.grid(True, alpha=0.3)
    
    # Score vs Difficulty
    ax = axes[1, 0]
    for diff in range(1, 6):
        diff_scores = [r.score for i, r in enumerate(session.responses) 
                      if session.questions[i].difficulty == diff]
        if diff_scores:
            ax.scatter([diff] * len(diff_scores), diff_scores, alpha=0.5, s=80)
    ax.axhline(y=0.6, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel('Difficulty level')
    ax.set_ylabel('Score')
    ax.set_title('Score by difficulty')
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.grid(True, alpha=0.3)
    
    # Summary metrics
    ax = axes[1, 1]
    metrics = ['Mean\\nscore', 'Pass\\nrate']
    values = [np.mean(scores), sum(r.correct for r in session.responses)/len(session.responses)]
    colors_bar = ['green' if v >= 0.6 else 'red' for v in values]
    bars = ax.bar(metrics, values, color=colors_bar, alpha=0.7)
    ax.set_ylabel('Value')
    ax.set_title('Summary metrics')
    ax.set_ylim(0, 1.05)
    ax.axhline(y=0.6, color='red', linestyle='--', alpha=0.5)
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height, f'{val:.2f}',
                ha='center', va='bottom')
    
    plt.tight_layout()
    plt.savefig('adaptive_test_results.png', dpi=300, bbox_inches='tight')
    plt.show()
    
print('Plotting function defined')

In [ ]:
# Execute adaptive test
session = TestSession(
    subject=SUBJECT_AREA,
    student_model=STUDENT_MODEL,
    teacher_model=TEACHER_MODEL
)

results = session.run(
    max_questions=MAX_QUESTIONS,
    fail_threshold=FAIL_THRESHOLD
)

In [ ]:
# Statistical analysis
scores = [r.score for r in session.responses]

print('\\nANALYSIS SUMMARY')
print('='*60)
print(f'Total questions: {len(session.responses)}')
print(f'Mean score: {np.mean(scores):.3f}')
print(f'Std dev: {np.std(scores):.3f}')
print(f'Pass rate: {sum(r.correct for r in session.responses)/len(session.responses)*100:.1f}%')

if session.failure_point is not None:
    fail_q = session.questions[session.failure_point]
    print(f'\\nKnowledge boundary:')
    print(f'  Question: {session.failure_point+1}')
    print(f'  Topic: {fail_q.topic}')
    print(f'  Difficulty: {fail_q.difficulty}/5')
    print(f'  Score: {session.responses[session.failure_point].score:.3f}')

In [ ]:
# Generate plots
plot_results(session)